
# Mental Health Late Fusion Model

Notebook ini menggunakan:
- Deep Learning untuk data tabular
- NLP untuk data teks
- Late Fusion Adjustment
- TensorFlow Functional API
- Custom Layer

Tanpa Transformer / IndoBERT.


In [ ]:

# =========================================================
# 1. IMPORT LIBRARY
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Input,
    Embedding,
    LSTM,
    Bidirectional
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

print("TensorFlow Version:", tf.__version__)


In [ ]:

# =========================================================
# 2. LOAD TABULAR DATASET
# =========================================================

tabular_df = pd.read_csv("survey.csv")

tabular_df = tabular_df.fillna("Unknown")

drop_cols = ["Timestamp", "comments", "state"]

for col in drop_cols:
    if col in tabular_df.columns:
        tabular_df = tabular_df.drop(columns=col)

print(tabular_df.head())
print(tabular_df.columns)


In [ ]:

# =========================================================
# 3. LOAD TEXT DATASET
# =========================================================

text_df = pd.read_csv("mental_health.csv")

print(text_df.head())
print(text_df.columns)


In [ ]:

# =========================================================
# 4. PREPROCESS TABULAR DATA
# =========================================================

target_tab = "treatment"

X_tab = tabular_df.drop(columns=[target_tab])
y_tab = tabular_df[target_tab]

encoders = {}

for col in X_tab.columns:

    le = LabelEncoder()

    X_tab[col] = le.fit_transform(
        X_tab[col].astype(str)
    )

    encoders[col] = le

label_tab = LabelEncoder()

y_tab = label_tab.fit_transform(y_tab)

y_tab_onehot = to_categorical(y_tab)

scaler = StandardScaler()

X_tab = scaler.fit_transform(X_tab)

X_tab_train, X_tab_test, y_tab_train, y_tab_test = train_test_split(
    X_tab,
    y_tab_onehot,
    test_size=0.2,
    random_state=42
)

print(X_tab_train.shape)


In [ ]:

# =========================================================
# 5. PREPROCESS TEXT DATA
# =========================================================

TEXT_COLUMN = "text"
LABEL_COLUMN = "status"

X_text = text_df[TEXT_COLUMN].astype(str)

label_txt = LabelEncoder()

y_text = label_txt.fit_transform(
    text_df[LABEL_COLUMN]
)

y_txt_onehot = to_categorical(y_text)

X_txt_train, X_txt_test, y_txt_train, y_txt_test = train_test_split(
    X_text,
    y_txt_onehot,
    test_size=0.2,
    random_state=42
)

print(label_txt.classes_)


In [ ]:

# =========================================================
# 6. CUSTOM LAYER
# =========================================================

class AttentionLayer(layers.Layer):

    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):

        self.W = self.add_weight(
            shape=(input_shape[-1], 1),
            initializer="random_normal",
            trainable=True
        )

    def call(self, inputs):

        score = tf.nn.tanh(
            tf.matmul(inputs, self.W)
        )

        weights = tf.nn.softmax(score, axis=1)

        context = weights * inputs

        context = tf.reduce_sum(
            context,
            axis=1
        )

        return context


In [ ]:
# =========================================================
# 7. TABULAR MODEL
# =========================================================

input_tab = layers.Input(shape=(X_tab_train.shape[1],), name="input_tab")

x1 = layers.Dense(64, activation="relu")(input_tab)

x1 = layers.Dropout(0.3)(x1)

x1 = layers.Dense(32, activation="relu")(x1)

out_tab = layers.Dense(2, activation="softmax", name="out_tab")(x1)

model_tab = models.Model(input_tab, out_tab)

model_tab.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

model_tab.summary()

In [ ]:
# =========================================================
# 8. TRAIN TABULAR MODEL
# =========================================================

history_tab = model_tab.fit(
    X_tab_train,
    y_tab_train,
    validation_data=(X_tab_test, y_tab_test),
    epochs=10,
    batch_size=32,
)

In [ ]:
# =========================================================
# 9. NLP MODEL
# =========================================================

vectorize_layer = layers.TextVectorization(
    max_tokens=5000, output_mode="int", output_sequence_length=50
)

vectorize_layer.adapt(X_txt_train)

input_txt = layers.Input(shape=(1,), dtype=tf.string, name="input_txt")

x2 = vectorize_layer(input_txt)

x2 = layers.Embedding(5000, 64)(x2)

x2 = Bidirectional(LSTM(64, return_sequences=True))(x2)

x2 = AttentionLayer()(x2)

x2 = layers.Dropout(0.3)(x2)

out_txt = layers.Dense(4, activation="softmax", name="out_txt")(x2)

model_txt = models.Model(input_txt, out_txt)

model_txt.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

model_txt.summary()

In [ ]:

# =========================================================
# 10. TRAIN NLP MODEL
# =========================================================

history_txt = model_txt.fit(
    np.array(X_txt_train).reshape(-1,1),
    y_txt_train,
    validation_data=(
        np.array(X_txt_test).reshape(-1,1),
        y_txt_test
    ),
    epochs=10,
    batch_size=32
)


In [ ]:
# =========================================================
# 11. EVALUATE TABULAR MODEL
# =========================================================

loss_tab, acc_tab = model_tab.evaluate(X_tab_test, y_tab_test)

print("Tabular Accuracy:", acc_tab)

In [ ]:

# =========================================================
# 12. EVALUATE NLP MODEL
# =========================================================

loss_txt, acc_txt = model_txt.evaluate(
    np.array(X_txt_test).reshape(-1,1),
    y_txt_test
)

print("Text Accuracy:", acc_txt)


In [ ]:

# =========================================================
# 13. SAMPLE INFERENCE
# =========================================================

X_tabular = X_tab_test[:3]

X_text = np.array([
    ["i feel anxious and depressed lately"],
    ["i am happy and calm"],
    ["i want to end my life"]
])

prob_tab = model_tab.predict(X_tabular)
prob_txt = model_txt.predict(X_text)

print("Probabilitas Tabular (No Stress | Stress):\n")
print(prob_tab)

print("\nProbabilitas Teks (anxiety | depression | normal | suicidal):\n")
print(prob_txt)


In [ ]:

# =========================================================
# 14. LATE FUSION ADJUSTMENT
# =========================================================

w_tab = 0.4
w_txt = 0.6

stress_score_tab = prob_tab[:, 1]

adjustment = np.stack([

    stress_score_tab,
    stress_score_tab,
    1 - stress_score_tab,
    stress_score_tab

], axis=1)

adjustment = adjustment / adjustment.sum(
    axis=1,
    keepdims=True
)

final_prob = (
    (w_tab * adjustment)
    +
    (w_txt * prob_txt)
)

final_prob = final_prob / final_prob.sum(
    axis=1,
    keepdims=True
)

final_class = np.argmax(
    final_prob,
    axis=1
)

label_map = {
    0: "anxiety",
    1: "depression",
    2: "normal",
    3: "suicidal"
}


In [ ]:

# =========================================================
# 15. FINAL RESULT
# =========================================================

print("\n--- HASIL LATE FUSION ---\n")

for i in range(len(final_class)):

    print(
        f"Sample {i+1}: "
        f"{label_map[final_class[i]]:12s} | "
        f"prob={final_prob[i].round(3)} | "
        f"stress={stress_score_tab[i]:.2f}"
    )

print("\nPrediksi kelas akhir:")
print(final_class)


In [ ]:

# =========================================================
# 16. SAVE MODEL
# =========================================================

model_tab.save("tabular_model.keras")

model_txt.save("text_model.keras")

print("Model berhasil disimpan!")


In [ ]:

# =========================================================
# 17. LOAD MODEL
# =========================================================

loaded_tab = tf.keras.models.load_model(
    "tabular_model.keras"
)

loaded_txt = tf.keras.models.load_model(
    "text_model.keras",
    custom_objects={
        "AttentionLayer": AttentionLayer
    }
)

print("Model berhasil di-load kembali!")
